# Amazon Bedrock AgentCore Memory로 Memory Record Event Streaming

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Memory의 [**Memory Record Streaming**](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-record-streaming.html)을 설정하는 방법을 살펴봅니다. Memory Record가 생성, 업데이트 또는 삭제될 때 실시간 알림을 받도록 [Amazon Kinesis Data Stream](https://docs.aws.amazon.com/streams/latest/dev/introduction.html)을 구성하여 API polling 없는 event-driven 아키텍처를 구현합니다.

### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                          |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | Memory Record Streaming                                          |
| 기능             | 장기 메모리 Event Streaming                                 |
| 주요 기능        | Kinesis Data Streams, Memory Record 수명 주기 Event             |
| 예제 난이도  | 중급                                                     |
| 사용 SDK            | boto3                                                            |

### 학습 내용

이 튜토리얼에서는 다음 방법을 학습합니다.
1. Memory Record event를 수신할 Amazon Kinesis Data Stream 생성
2. AgentCore Memory가 stream에 게시할 수 있도록 IAM 역할 설정
3. Streaming이 활성화된 Memory 리소스 생성
4. Memory Record 수명 주기 event를 실시간으로 trigger하고 consume
5. Event 콘텐츠 수준(`FULL_CONTENT` 또는 `METADATA_ONLY`) 구성
   
### 작동 방식

Memory Record Streaming은 push 기반 전송 모델을 사용합니다. Memory Record가 변경되면 event가 Kinesis Data Stream에 자동으로 게시됩니다.

- **MemoryRecordCreated** - 장기 메모리 추출 또는 `BatchCreateMemoryRecords` API로 trigger
- **MemoryRecordUpdated** - `BatchUpdateMemoryRecords` API로 trigger
- **MemoryRecordDeleted** - 통합 workflow, `DeleteMemoryRecord` 또는 `BatchDeleteMemoryRecords` API로 trigger

### 아키텍처

<div style="text-align:left">
    <img src="memory_record_streaming.png" width="90%"/>
</div>

## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* AgentCore Memory, Kinesis 및 IAM 액세스 권한이 구성된 AWS 자격 증명
* Amazon Bedrock 모델 액세스(장기 메모리 추출용)

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install boto3>=1.42.63

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성합니다.

In [ ]:
import os
import json
import time
import uuid
import base64
import logging
import boto3
from datetime import datetime, timezone
from botocore.exceptions import ClientError

# 구성
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("memory-streaming")

REGION = os.getenv("AWS_REGION", "us-west-2")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# boto3 client 초기화
kinesis_client = boto3.client("kinesis", region_name=REGION)
iam_client = boto3.client("iam")
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# 리소스 이름을 위한 고유 식별자
unique_id = str(uuid.uuid4())[:8]
print(f"Account: {ACCOUNT_ID}, Region: {REGION}, Unique ID: {unique_id}")

## 1. Kinesis Data Stream 생성

먼저 계정에 Kinesis Data Stream을 생성합니다. AgentCore Memory는 이 stream에 Memory Record 수명 주기 event를 게시합니다.

이 튜토리얼에는 초당 최대 1,000개의 record 쓰기를 지원하는 단일 shard로 충분합니다. 프로덕션 workload의 용량을 확장하려면 [Stream Resharding](https://docs.aws.amazon.com/streams/latest/dev/kinesis-using-sdk-java-resharding.html)을 확인하세요.

In [ ]:
stream_name = f"memory-record-stream-{unique_id}"

try:
    kinesis_client.create_stream(
        StreamName=stream_name,
        ShardCount=1,  # 이 튜토리얼에는 단일 shard로 충분
    )
    logger.info(f"Creating Kinesis stream: {stream_name}")

    # Stream이 활성화될 때까지 대기
    waiter = kinesis_client.get_waiter("stream_exists")
    waiter.wait(StreamName=stream_name)

    stream_info = kinesis_client.describe_stream(StreamName=stream_name)
    stream_arn = stream_info["StreamDescription"]["StreamARN"]
    print(f"Kinesis stream created: {stream_arn}")

except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceInUseException":
        stream_info = kinesis_client.describe_stream(StreamName=stream_name)
        stream_arn = stream_info["StreamDescription"]["StreamARN"]
        print(f"Stream already exists: {stream_arn}")
    else:
        raise

## 2. Streaming용 IAM 역할 생성

AgentCore Memory가 Kinesis Data Stream에 event를 게시하려면 assume할 수 있는 IAM 역할이 필요합니다. 이 역할에는 다음이 필요합니다.
- `bedrock-agentcore.amazonaws.com`이 역할을 assume할 수 있도록 허용하는 **trust policy**
- stream에 대한 `kinesis:PutRecords` 및 `kinesis:DescribeStream`을 허용하는 **permissions policy**

In [ ]:
role_name = f"AgentCoreMemoryStreamingRole-{unique_id}"

# Trust policy - AgentCore Memory가 이 역할을 assume하도록 허용
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# Permissions policy - 지정한 Kinesis stream으로 범위 제한
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["kinesis:PutRecords", "kinesis:DescribeStream"],
            "Resource": stream_arn,
        }
    ],
}

try:
    # IAM 역할 생성
    create_role_response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Allows AgentCore Memory to publish events to Kinesis",
    )
    role_arn = create_role_response["Role"]["Arn"]

    # Inline permissions policy 연결
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName="KinesisPublishPolicy",
        PolicyDocument=json.dumps(permissions_policy),
    )

    # IAM propagation 시간 확보
    print(f"IAM role created: {role_arn}")
    print("Waiting 10 seconds for IAM propagation...")
    time.sleep(10)

except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{role_name}"
        print(f"Role already exists: {role_arn}")
    else:
        raise

## 3. Streaming이 활성화된 Memory 생성

이제 stream 전송 구성이 포함된 AgentCore Memory 리소스를 생성합니다. 주요 parameter는 다음과 같습니다.

- **`streamDeliveryResources`** - Kinesis stream을 가리키며 콘텐츠 수준을 지정
- **`memoryExecutionRoleArn`** - AgentCore가 event 게시를 위해 assume할 IAM 역할
- **`FULL_CONTENT`** - 각 event에 Memory Record 텍스트 포함(경량 알림에는 `METADATA_ONLY` 사용)

또한 대화 event를 Memory Record로 추출하고 이어서 streaming event를 trigger하도록 장기 메모리 strategy(사용자 선호도)를 구성합니다.

In [ ]:
memory_name = "streaming_memory"
actor_id = "demo-user"


def wait_for_memory_active(memory_id, timeout=250):
    """메모리 상태가 ACTIVE가 되거나 시간이 초과될 때까지 GetMemory를 폴링합니다."""
    start = time.time()
    while time.time() - start < timeout:
        resp = agentcore_control_client.get_memory(memoryId=memory_id)
        status = resp["memory"]["status"]
        print(f"  Memory status: {status}")
        if status == "ACTIVE":
            return resp["memory"]
        if status == "FAILED":
            raise RuntimeError(f"Memory creation failed: {resp['memory'].get('failureReason')}")
        time.sleep(5)
    raise TimeoutError("Timed out waiting for memory to become ACTIVE")


try:
    response = agentcore_control_client.create_memory(
        name=memory_name,
        description="Memory with long-term memory record streaming enabled",
        eventExpiryDuration=7,
        memoryExecutionRoleArn=role_arn,
        memoryStrategies=[
            {
                "userPreferenceMemoryStrategy": {
                    "name": "UserPreferences",
                    "description": "Extracts user preferences, facts, and interests from conversations",
                    "namespaceTemplates": ["/{actorId}/user_preferences/"],
                }
            }
        ],
        streamDeliveryResources={
            "resources": [
                {
                    "kinesis": {
                        "dataStreamArn": stream_arn,
                        "contentConfigurations": [{"type": "MEMORY_RECORDS", "level": "FULL_CONTENT"}],
                    }
                }
            ]
        },
    )
    memory_id = response["memory"]["id"]
    print(f"Memory creation initiated: {memory_id}")
    print("Waiting for memory to become ACTIVE...")
    wait_for_memory_active(memory_id)
    print(f"Memory created with streaming enabled: {memory_id}")

except ClientError as e:
    logger.error(f"Error creating memory: {e}")
    raise

## 4. Streaming 활성화 확인

Streaming이 포함된 Memory를 생성하면 AgentCore Memory가 구성을 검증하고 Kinesis stream에 `StreamingEnabled` event를 게시합니다. Stream을 읽어 확인해 보겠습니다.

In [ ]:
def read_kinesis_events(stream_name, max_wait_seconds=60, max_events=10):
    """Kinesis Data Stream에서 이벤트를 읽습니다.

    새 레코드가 있는지 스트림을 폴링하고 디코딩합니다.

    인자:
        stream_name: Kinesis 스트림 이름
        max_wait_seconds: 반환 전까지 폴링할 최대 시간
        max_events: 수집할 최대 이벤트 수

    반환값:
        디코딩된 이벤트 페이로드 목록
    """
    events = []

    # 가장 오래된 record부터 시작하는 shard iterator 가져오기
    stream_info = kinesis_client.describe_stream(StreamName=stream_name)
    shard_id = stream_info["StreamDescription"]["Shards"][0]["ShardId"]

    iterator_response = kinesis_client.get_shard_iterator(
        StreamName=stream_name,
        ShardId=shard_id,
        ShardIteratorType="TRIM_HORIZON",  # 가장 오래된 record부터 읽기
    )
    shard_iterator = iterator_response["ShardIterator"]

    start_time = time.time()
    while time.time() - start_time < max_wait_seconds and len(events) < max_events:
        response = kinesis_client.get_records(ShardIterator=shard_iterator, Limit=100)

        for record in response["Records"]:
            data = base64.b64decode(record["Data"]) if isinstance(record["Data"], str) else record["Data"]
            payload = json.loads(data)
            events.append(payload)

        shard_iterator = response["NextShardIterator"]

        if not response["Records"]:
            time.sleep(2)

    return events

In [ ]:
# StreamingEnabled 검증 event 확인
print("Checking for StreamingEnabled event...")
events = read_kinesis_events(stream_name, max_wait_seconds=30, max_events=1)

if events:
    for event in events:
        print(json.dumps(event, indent=2))
else:
    print("No events received yet. The StreamingEnabled event may take a moment to arrive.")

## 5. Memory Record Event trigger

이제 대화 데이터를 생성하여 Memory Record 수명 주기 event를 발생시킵니다. 두 가지 방법을 사용합니다.

| 접근 방식 | API | 작동 방식 |
|:---------|:----|:-------------|
| **옵션 A** | [`CreateEvent`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_CreateEvent.html) | 대화를 전송하면 AgentCore가 구성된 strategy로 장기 record를 **비동기적으로** 추출 |
| **옵션 B** | [`BatchCreateMemoryRecords`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_BatchCreateMemoryRecords.html) | Record를 직접 생성하면 event가 **즉시** 게시됨 |

### 옵션 A: 단기 메모리를 통해 event 생성(비동기 추출 trigger)

대화 event를 전송하면 AgentCore Memory는 구성된 strategy로 장기 Memory Record를 비동기적으로 추출합니다. 추출된 각 record는 stream에서 `MemoryRecordCreated` event를 trigger합니다.

In [ ]:
# 추출 가능한 사용자 선호도가 포함된 대화 전송
agentcore_client.create_event(
    memoryId=memory_id,
    actorId=f"{actor_id}",
    sessionId="streaming-demo-session-001",
    eventTimestamp=datetime.now(timezone.utc),
    payload=[
        {
            "conversational": {
                "content": {
                    "text": "I went hiking yesterday. I really like hiking in the Pacific Northwest. I also enjoyed the Thai restaurant. Thai food is amazing."
                },
                "role": "USER",
            }
        },
        {
            "conversational": {
                "content": {
                    "text": "Great! I'll remember that you enjoy hiking in the Pacific Northwest and prefer Thai dining options."
                },
                "role": "ASSISTANT",
            }
        },
    ],
)

print("Conversation event sent. Long-term memory extraction will happen asynchronously.")

### 옵션 B: Memory Record 직접 생성

`BatchCreateMemoryRecords` API로 Memory Record를 직접 생성할 수도 있습니다. 생성된 각 record는 즉시 `MemoryRecordCreated` event를 trigger합니다.

In [ ]:
response = agentcore_client.batch_create_memory_records(
    memoryId=memory_id,
    records=[
        {
            "requestIdentifier": "direct-record-1",
            "content": {"text": "User prefers window seats on flights"},
            "namespaces": [f"/{actor_id}/user_preferences/"],
            "timestamp": str(int(time.time())),
        },
        {
            "requestIdentifier": "direct-record-2",
            "content": {"text": "User's favorite programming language is Python"},
            "namespaces": [f"/{actor_id}/user_preferences/"],
            "timestamp": str(int(time.time())),
        },
    ],
)

print(f"Batch created {len(response.get('successfulRecords', []))} memory records directly.")
print(f"with record IDs: \n {[i.get('memoryRecordId') for i in response.get('successfulRecords', [])]}")

## 6. Streaming Event consume

Kinesis stream을 읽어 Memory Record 수명 주기 event를 확인해 보겠습니다. 장기 메모리 추출은 비동기 방식이므로 처리 시간을 확보하기 위해 최대 90초 동안 polling합니다.


> **프로덕션 참고:** 실제 애플리케이션에서는 polling 대신 [AWS Lambda event source mapping](https://docs.aws.amazon.com/lambda/latest/dg/with-kinesis.html) 또는 [Amazon Kinesis Client Library(KCL)](https://docs.aws.amazon.com/streams/latest/dev/shared-throughput-kcl-consumers.html)를 사용하여 stream을 consume할 수 있습니다.

In [ ]:
print("Polling Kinesis stream for memory record events...\n")
events = read_kinesis_events(stream_name, max_wait_seconds=90, max_events=10)

print(f"Received {len(events)} event(s):\n")
for i, event in enumerate(events):
    stream_event = event.get("memoryStreamEvent", {})
    event_type = stream_event.get("eventType", "Unknown")
    event_time = stream_event.get("eventTime", "")
    record_id = stream_event.get("memoryRecordId", "N/A")
    record_text = stream_event.get("memoryRecordText", "")

    print(f"--- Event {i + 1}: {event_type} ---")
    print(f"  Time:      {event_time}")
    print(f"  Memory ID: {stream_event.get('memoryId', 'N/A')}")
    print(f"  Record ID: {record_id}")
    if record_text:
        print(f"  Content:   {record_text[:120]}...")
    print()

### 전체 event payload 살펴보기

Event 하나의 원시 JSON을 살펴보고 전체 schema를 확인합니다.

In [ ]:
if events:
    # 첫 번째 MemoryRecordCreated event의 전체 payload 표시
    created_events = [e for e in events if e.get("memoryStreamEvent", {}).get("eventType") == "MemoryRecordCreated"]
    if created_events:
        print("Full MemoryRecordCreated event payload:")
        print(json.dumps(created_events[0], indent=2))
    else:
        print("Full payload of first event:")
        print(json.dumps(events[0], indent=2))
else:
    print("No events to inspect. Extraction may still be in progress — try re-running this cell.")

## 7. ListMemoryRecords와 상호 참조

Record를 직접 나열하여 streaming된 event가 Memory에 저장된 내용과 일치하는지 검증합니다.

In [ ]:
records_response = agentcore_client.list_memory_records(memoryId=memory_id, namespace=f"/{actor_id}/user_preferences/")

records = records_response.get("memoryRecordSummaries", [])
print(f"Found {len(records)} memory record(s) in namespace '/{actor_id}/user-preferences/':\n")

for record in records:
    print(f"  Record ID: {record['memoryRecordId']}")
    print(f"  Content:   {record.get('content', {}).get('text', 'N/A')[:120]}")
    print(f"  Created:   {record.get('createdAt', 'N/A')}")
    print()

## 8. 리소스 정리(선택 사항)

실습을 마치면 이 튜토리얼에서 생성한 리소스를 정리합니다.

> **비용 참고:** Kinesis Data Streams에는 [shard당 시간 요금](https://aws.amazon.com/kinesis/data-streams/pricing/)이 부과됩니다. 지속적인 비용을 방지하려면 실습을 마친 후 stream을 삭제하세요.

In [ ]:
# Memory 리소스 삭제
try:
    agentcore_control_client.delete_memory(memoryId=memory_id)
    print(f"Deleting memory: {memory_id}")
    # 삭제가 완료될 때까지 polling
    while True:
        try:
            resp = agentcore_control_client.get_memory(memoryId=memory_id)
            status = resp["memory"]["status"]
            print(f"  Memory status: {status}")
            if status == "DELETING":
                time.sleep(5)
            else:
                break
        except agentcore_control_client.exceptions.ResourceNotFoundException:
            print("  Memory deleted successfully.")
            break
except Exception as e:
    print(f"Error deleting memory: {e}")

# Kinesis stream 삭제
try:
    kinesis_client.delete_stream(StreamName=stream_name, EnforceConsumerDeletion=True)
    print(f"Deleted Kinesis stream: {stream_name}")
except Exception as e:
    print(f"Error deleting stream: {e}")

# IAM 역할 삭제(먼저 inline policy를 제거해야 함)
try:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName="KinesisPublishPolicy")
    iam_client.delete_role(RoleName=role_name)
    print(f"Deleted IAM role: {role_name}")
except Exception as e:
    print(f"Error deleting IAM role: {e}")

## 마무리

이 튜토리얼에서는 Amazon Bedrock AgentCore Memory로 end-to-end Memory Record Streaming을 설정했습니다. 학습한 내용은 다음과 같습니다.

1. Memory Record 수명 주기 event를 수신할 **Kinesis Data Stream 생성**
2. AgentCore가 stream에 게시할 수 있도록 최소 권한이 적용된 **IAM 역할 구성**
3. Streaming 및 `FULL_CONTENT` 전송이 활성화된 **Memory 리소스 생성**
4. 대화 추출과 직접 record 생성을 통한 **event trigger**
5. Stream의 `MemoryRecordCreated` event를 실시간으로 **consume하고 살펴보기**

### 다음 단계
AgentCore Memory Streaming 기능을 더 효과적으로 활용하려면 다음 내용을 시도해 보세요.
- event를 자동으로 처리하는 **Lambda consumer 추가**([문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/userguide/memory-streaming.html)의 예제 참조)
- 변경 알림만 필요할 때 데이터 전송량을 줄이도록 콘텐츠 수준을 **`METADATA_ONLY`로 전환**
- 프로덕션 모니터링을 위해 `StreamPublishingFailure` 및 `StreamUserError` metric에 **CloudWatch alarm 설정**
- Memory Record를 S3 data lake에 동기화하거나 알림을 trigger하고 downstream 사용자 프로필을 업데이트하는 **event-driven workflow 구축**